In [42]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Approximate haploid genome length (Mb)
GENOME_LENGTH_MB = 2880

PLINK_FILE = "./results/plink_results/full_plink.genome"
GERMLINE_FILE = "./results/germline_results/germline_full_out.match"

In [43]:
# ===============================
# PLINK Loader
# ===============================

def load_plink_results(filepath):

    if not os.path.exists(filepath):
        print("PLINK file not found")
        return pd.DataFrame()

    df = pd.read_csv(filepath, delim_whitespace=True)

    # PLINK genome files usually contain:
    # FID1 IID1 FID2 IID2 PI_HAT ...

    required_cols = ["IID1", "IID2", "PI_HAT"]

    # Try flexible column detection
    if "IID1" not in df.columns:
        # assume PLINK format with FID IID pairs
        df["IID1"] = df.iloc[:, 1].astype(str)
        df["IID2"] = df.iloc[:, 3].astype(str)
        df["PI_HAT"] = pd.to_numeric(df.iloc[:, 6], errors="coerce")

    df["pair_id"] = df.apply(
        lambda r: "_".join(sorted([r["IID1"], r["IID2"]])),
        axis=1
    )

    return df[["pair_id", "IID1", "IID2", "PI_HAT"]]

In [44]:
# ===============================
# GERMLINE Loader
# ===============================

def load_germline_results(filepath):

    if not os.path.exists(filepath) or os.path.getsize(filepath) == 0:
        print("GERMLINE output empty")
        return pd.DataFrame()

    df = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None,
        engine="python"
    )

    print("GERMLINE detected columns:", df.shape[1])

    # GERMLINE format assumption:
    # 0 IID1_FID
    # 1 IID1_IID
    # 2 IID2_FID
    # 3 IID2_IID
    # 4 chromosome
    # 5 start_bp
    # 6 end_bp
    # 7 SNP_start
    # 8 SNP_end
    # 9 segment_snps
    # 10 segment_length
    # 11 unit (MB or cM)

    if df.shape[1] < 11:
        print("Unexpected GERMLINE format")
        return pd.DataFrame()

    df["IID1"] = df.iloc[:, 0].astype(str)
    df["IID2"] = df.iloc[:, 2].astype(str)
    start = pd.to_numeric(df.iloc[:, 5], errors="coerce")
    end = pd.to_numeric(df.iloc[:, 6], errors="coerce")
    chr = pd.to_numeric(df.iloc[:, 4], errors="coerce")

    # Extract segment length safely
    length = pd.to_numeric(df.iloc[:, 10], errors="coerce")

    out = pd.DataFrame({
        "IID1": df["IID1"],
        "IID2": df["IID2"],
        "start": start,
        "end": end,
        "length": length,
        "chr": chr
    })
    print("out:\n")
    print(out)

    return out

In [45]:
# ===============================
# GERMLINE Relatedness Score
# ===============================

def compute_germline_relatedness(df):

    if df.empty:
        return pd.DataFrame(columns=["pair_id", "GERMLINE_score"])

    # Construct standardized pair ID
    df["pair_id"] = df.apply(
        lambda r: "_".join(sorted([str(r["IID1"]), str(r["IID2"])])),
        axis=1
    )

    # Convert genomic coordinates safely
    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["end"] = pd.to_numeric(df["end"], errors="coerce")
    df = df.dropna(subset=["start", "end"])

    # Group by pair + chromosome (VERY IMPORTANT ⭐)
    grouped = df.groupby(["pair_id", "chr"])

    def merge_intervals(group):

        group = group.sort_values("start")

        merged_length = 0

        curr_start = group.iloc[0]["start"]
        curr_end = group.iloc[0]["end"]

        for _, row in group.iloc[1:].iterrows():

            if row["start"] <= curr_end:
                curr_end = max(curr_end, row["end"])
            else:
                merged_length += curr_end - curr_start
                curr_start = row["start"]
                curr_end = row["end"]

        merged_length += curr_end - curr_start

        # Convert bp → Mb
        return merged_length / 1e6

    # Apply merging
    merged_records = grouped.apply(
        merge_intervals
    ).reset_index(name="merged_mb")

    # Sum across chromosomes
    pair_lengths = merged_records.groupby("pair_id")[
        "merged_mb"
    ].sum().reset_index()

    # Normalize relatedness score
    GENOME_LENGTH_CM = 3600

    pair_lengths["GERMLINE_score"] = (
        pair_lengths["merged_mb"] / GENOME_LENGTH_CM
    )

    print("pair lengths normalized\n")
    print(pair_lengths)

    return pair_lengths[["pair_id", "GERMLINE_score"]]


# ===============================
# Merge Results
# ===============================

def merge_results(plink_df, germline_df):

    print("PLINK pairs example:")
    print(plink_df["pair_id"].head())

    print("\nGERMLINE pairs example:")
    print(germline_df["pair_id"].head())

    merged = pd.merge(
        plink_df,
        germline_df,
        on="pair_id",
        how="inner"
    )

    return merged

In [46]:
# ===============================
# Analysis
# ===============================

def analyze_results(merged_df):

    print("\n===== Summary Statistics =====")

    if merged_df.empty or len(merged_df) < 2:
        print("Not enough data points for correlation analysis.")
        print(f"Matched pairs = {len(merged_df)}")
        return

    print("\nPLINK PI_HAT:")
    print(merged_df["PI_HAT"].describe())

    print("\nGERMLINE Score:")
    print(merged_df["GERMLINE_score"].describe())

    try:
        corr, pval = pearsonr(
            merged_df["PI_HAT"],
            merged_df["GERMLINE_score"]
        )

        print("\n===== Correlation Analysis =====")
        print(f"Pearson correlation = {corr:.4f}")
        print(f"p-value = {pval:.4e}")

    except Exception:
        print("Correlation computation failed")

# ===============================
# Plotting
# ===============================

def plot_scatter(merged_df):

    plt.figure(figsize=(8,6))

    plt.scatter(
        merged_df["PI_HAT"],
        merged_df["GERMLINE_score"],
        alpha=0.6
    )

    # Identity reference line
    plt.plot([0,1],[0,1], linestyle="--")

    plt.xlabel("PLINK PI_HAT")
    plt.ylabel("GERMLINE Relatedness Score")
    plt.title("PLINK vs GERMLINE Relatedness Comparison")

    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# ===============================
# Main
# ===============================

def main():

    print("Loading PLINK results...")
    plink_df = load_plink_results(PLINK_FILE)

    print("Loading GERMLINE results...")
    germline_raw = load_germline_results(GERMLINE_FILE)

    germline_df = compute_germline_relatedness(germline_raw)

    print("Merging datasets...")
    merged_df = merge_results(plink_df, germline_df)

    print(f"\nMatched pairs = {len(merged_df)}")

    analyze_results(merged_df)

    plot_scatter(merged_df)


if __name__ == "__main__":
    main()

Loading PLINK results...
Loading GERMLINE results...
